In [ ]:
# import shutil

# src_dir = "/content/drive/MyDrive/sprog_checkpoints/tinystories_run_v4"
# dst_dir = "/content/drive/MyDrive/sprog_checkpoints/tinystories_run_v6"

# # Copies the entire directory tree from v5 to v6
# shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)

# # Set the active checkpoint directory
# CHECKPOINT_DIR = dst_dir

In [ ]:
!pip install bitsandbytes

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

# Set up your persistent directory on Drive
CHECKPOINT_DIR = "/content/drive/MyDrive/sprog_checkpoints/tinystories_run_v6"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BIN_PATH = os.path.join(CHECKPOINT_DIR, "train.bin")
TOKEN_PROGRESS_PATH = os.path.join(CHECKPOINT_DIR, "tokenization_progress.txt")
CKPT_PATH = os.path.join(CHECKPOINT_DIR, "latest_ckpt.pt")

Mounted at /content/drive


In [ ]:
import os
import numpy as np
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from tqdm.auto import tqdm

# Load/initialize tokenizer
tokenizer = ByteLevelBPETokenizer()
# Ensure your tokenizer files are loaded or saved:
tokenizer = ByteLevelBPETokenizer.from_file(os.path.join(CHECKPOINT_DIR, "tinystories_run-vocab.json"), os.path.join(CHECKPOINT_DIR, "tinystories_run-merges.txt"))
eos_id = tokenizer.token_to_id("<|endoftext|>")
if eos_id is None:
    tokenizer.add_special_tokens(["<|endoftext|>"])
    eos_id = tokenizer.token_to_id("<|endoftext|>")

ds = load_dataset("roneneldan/TinyStories", split="train")
total_examples = len(ds)

# Check last processed index
start_idx = 0
if os.path.exists(TOKEN_PROGRESS_PATH) and os.path.exists(BIN_PATH):
    with open(TOKEN_PROGRESS_PATH, "r") as f:
        content = f.read().strip()
        if content.isdigit():
            start_idx = int(content)
            print(f"Resuming tokenization from example index {start_idx}/{total_examples}...")

CHUNK_SIZE = 10000  # Number of examples to process before flushing to Drive

if start_idx < total_examples:
    with open(BIN_PATH, "ab") as bin_file:  # Append binary mode
        buffer = []
        for i in range(start_idx, total_examples):
            text = ds[i]["text"]
            encoded = tokenizer.encode(text).ids
            encoded.append(eos_id)
            buffer.extend(encoded)

            # Flush chunk to Drive
            if (i + 1) % CHUNK_SIZE == 0 or (i + 1) == total_examples:
                chunk_arr = np.array(buffer, dtype=np.uint16)
                bin_file.write(chunk_arr.tobytes())
                bin_file.flush()
                os.fsync(bin_file.fileno())
                buffer = []

                # Update progress tracker atomically
                with open(TOKEN_PROGRESS_PATH + ".tmp", "w") as f:
                    f.write(str(i + 1))
                os.replace(TOKEN_PROGRESS_PATH + ".tmp", TOKEN_PROGRESS_PATH)
                print(f"Flushed up to index {i + 1}/{total_examples}")

    print("Tokenization and packing complete.")
else:
    print(f"Binary file is already complete ({total_examples}/{total_examples} examples).")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Resuming tokenization from example index 2119719/2119719...
Binary file is already complete (2119719/2119719 examples).


In [ ]:
# import math
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class RMSNorm(nn.Module):
#     def __init__(self, dim: int, eps: float = 1e-6):
#         super().__init__()
#         self.eps = eps
#         self.weight = nn.Parameter(torch.ones(dim))

#     def forward(self, x):
#         norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
#         return x * norm * self.weight


# class CausalSelfAttention(nn.Module):
#     def __init__(self, n_embd: int, n_head: int):
#         super().__init__()
#         assert n_embd % n_head == 0
#         self.n_head = n_head
#         self.head_dim = n_embd // n_head
#         self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
#         self.out_proj = nn.Linear(n_embd, n_embd, bias=False)

#     def forward(self, x):
#         B, T, C = x.size()
#         qkv = self.qkv_proj(x).reshape(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
#         q, k, v = qkv[0], qkv[1], qkv[2]

#         # PyTorch Native Scaled Dot-Product Attention
#         out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
#         out = out.transpose(1, 2).contiguous().view(B, T, C)
#         return self.out_proj(out)


# class MLP(nn.Module):
#     """Standard SwiGLU expert block."""
#     def __init__(self, n_embd: int):
#         super().__init__()
#         hidden_dim = 4 * n_embd
#         self.gate_proj = nn.Linear(n_embd, hidden_dim, bias=False)
#         self.up_proj = nn.Linear(n_embd, hidden_dim, bias=False)
#         self.down_proj = nn.Linear(hidden_dim, n_embd, bias=False)

#     def forward(self, x):
#         return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


# class MoE(nn.Module):
#     """Sparse Mixture of Experts with Top-K routing."""
#     def __init__(self, n_embd: int, num_experts: int = 8, top_k: int = 2):
#         super().__init__()
#         self.num_experts = num_experts
#         self.top_k = top_k
#         self.gate = nn.Linear(n_embd, num_experts, bias=False)
#         self.experts = nn.ModuleList([MLP(n_embd) for _ in range(num_experts)])

#     def forward(self, x):
#         # x shape: (B, T, C)
#         orig_shape = x.shape
#         x_flat = x.view(-1, orig_shape[-1])  # (B*T, C)
#         num_tokens, hidden_dim = x_flat.size()

#         # Compute gating logits and Top-K weights
#         gate_logits = self.gate(x_flat)  # (B*T, num_experts)
#         topk_logits, topk_indices = torch.topk(gate_logits, self.top_k, dim=-1)  # (B*T, top_k)
#         topk_weights = F.softmax(topk_logits, dim=-1)  # (B*T, top_k)

#         out = torch.zeros_like(x_flat)

#         # Vectorized routing loop across all experts
#         for expert_idx, expert in enumerate(self.experts):
#             # Mask identifying positions where this expert was selected
#             batch_idx, k_idx = torch.where(topk_indices == expert_idx)
#             if batch_idx.numel() == 0:
#                 continue

#             # Extract selected tokens, compute through expert, weight, and accumulate
#             tokens = x_flat[batch_idx]
#             expert_out = expert(tokens)
#             weights = topk_weights[batch_idx, k_idx].unsqueeze(-1)
#             out.index_add_(0, batch_idx, expert_out * weights)

#         return out.view(orig_shape)


# class TransformerBlock(nn.Module):
#     def __init__(self, n_embd: int, n_head: int, num_experts: int = 8, top_k: int = 2):
#         super().__init__()
#         self.attn_norm = RMSNorm(n_embd)
#         self.attn = CausalSelfAttention(n_embd, n_head)
#         self.moe_norm = RMSNorm(n_embd)
#         self.moe = MoE(n_embd, num_experts=num_experts, top_k=top_k)

#     def forward(self, x):
#         x = x + self.attn(self.attn_norm(x))
#         x = x + self.moe(self.moe_norm(x))
#         return x


# class CustomMoEGePrTr(nn.Module):
#     def __init__(
#         self,
#         vocab_size: int = 32000,
#         n_layer: int = 8,
#         n_head: int = 8,
#         n_embd: int = 512,
#         block_size: int = 512,
#         num_experts: int = 8,
#         top_k: int = 2,
#     ):
#         super().__init__()
#         self.block_size = block_size
#         self.tok_emb = nn.Embedding(vocab_size, n_embd)
#         self.pos_emb = nn.Embedding(block_size, n_embd)
#         self.blocks = nn.ModuleList([
#             TransformerBlock(n_embd, n_head, num_experts=num_experts, top_k=top_k)
#             for _ in range(n_layer)
#         ])
#         self.norm = RMSNorm(n_embd)
#         self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

#         # Weight tying
#         self.tok_emb.weight = self.lm_head.weight

#     def forward(self, idx):
#         B, T = idx.size()
#         pos = torch.arange(0, T, dtype=torch.long, device=idx.device)

#         x = self.tok_emb(idx) + self.pos_emb(pos)
#         for block in self.blocks:
#             x = block(x)
#         x = self.norm(x)
#         logits = self.lm_head(x)
#         return logits

In [ ]:
import os
import math
import time
import random
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb

from torch.optim.lr_scheduler import LambdaLR

# Enable CUDNN auto-tuner for static tensor shapes on Turing architecture
torch.backends.cudnn.benchmark = True


# ============================================================
# ARCHITECTURE DEFINITION
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class MLP(nn.Module):
    """Standard SwiGLU expert block."""
    def __init__(self, n_embd: int):
        super().__init__()
        hidden_dim = 4 * n_embd
        self.gate_proj = nn.Linear(n_embd, hidden_dim, bias=False)
        self.up_proj = nn.Linear(n_embd, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, n_embd, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class MoE(nn.Module):
    """Sparse Mixture of Experts with Top-K routing & Auxiliary Loss."""
    def __init__(self, n_embd: int, num_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([MLP(n_embd) for _ in range(num_experts)])

    def forward(self, x):
        orig_shape = x.shape
        x_flat = x.view(-1, orig_shape[-1])  # (B*T, C)

        # Gate logits: compute in FP32 for numerical stability under FP16 AMP
        gate_logits = self.gate(x_flat).float()  # (B*T, num_experts)
        gate_probs = F.softmax(gate_logits, dim=-1)  # (B*T, num_experts)

        # Top-K selection
        topk_weights, topk_indices = torch.topk(gate_probs, self.top_k, dim=-1)  # (B*T, top_k)

        # Renormalize weights across selected experts
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)
        topk_weights = topk_weights.to(x.dtype)

        # Auxiliary Load Balancing Loss (Switch/ST-MoE formulation)
        if self.training:
            me = gate_probs.mean(dim=0)  # Mean expert probability across tokens
            mask = F.one_hot(topk_indices, num_classes=self.num_experts).sum(dim=1)
            ce = mask.float().mean(dim=0)  # Fraction of tokens routed to each expert
            aux_loss = self.num_experts * torch.sum(me * ce)
        else:
            aux_loss = torch.tensor(0.0, device=x.device)

        out = torch.zeros_like(x_flat)

        # Expert dispatch and accumulation
        for expert_idx, expert in enumerate(self.experts):
            batch_idx, k_idx = torch.where(topk_indices == expert_idx)
            if batch_idx.numel() == 0:
                continue

            tokens = x_flat[batch_idx]
            expert_out = expert(tokens)
            weights = topk_weights[batch_idx, k_idx].unsqueeze(-1)
            out.index_add_(0, batch_idx, expert_out * weights)

        return out.view(orig_shape), aux_loss


class TransformerBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int, num_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.attn_norm = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.moe_norm = RMSNorm(n_embd)
        self.moe = MoE(n_embd, num_experts=num_experts, top_k=top_k)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        moe_out, aux_loss = self.moe(self.moe_norm(x))
        x = x + moe_out
        return x, aux_loss


class CustomMoEGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int = 32000,
        n_layer: int = 8,
        n_head: int = 8,
        n_embd: int = 512,
        block_size: int = 512,
        num_experts: int = 4,
        top_k: int = 2,
    ):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([
            TransformerBlock(n_embd, n_head, num_experts=num_experts, top_k=top_k)
            for _ in range(n_layer)
        ])
        self.norm = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying
        self.tok_emb.weight = self.lm_head.weight

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        total_aux_loss = torch.tensor(0.0, device=idx.device)
        for block in self.blocks:
            x, aux_loss = block(x)
            total_aux_loss = total_aux_loss + aux_loss

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits, total_aux_loss


# ============================================================
# CONFIGURATION & HYPERPARAMETERS
# ============================================================

BIN_PATH = BIN_PATH
LOCAL_BIN_PATH = "/content/train.bin"

CKPT_DIR = CHECKPOINT_DIR
CKPT_PATH = os.path.join(CKPT_DIR, "latest.ckpt")

VOCAB_SIZE = 32000
N_LAYER = 8
N_HEAD = 8
N_EMBD = 512
BLOCK_SIZE = 512

NUM_EXP = 4
TOP_K = 2
AUX_LOSS_COEF = 0.01

# Tuned batching for 16 GB Tesla T4 (~11-13 GB VRAM target)
# BATCH_SIZE = 32
# GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch = 64 sequences (32,768 tokens/step)
# This lowers peak VRAM usage to $\approx 7.5\text{–}8.5\text{ GB}$, leaving over $6\text{ GB}$ of headroom:
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch = 16 * 4 = 64 sequences (same as before)

TOTAL_STEPS = 50000
WARMUP_STEPS = 2000
SAVE_INTERVAL = 500
LOG_INTERVAL = 100

BASE_LR = 6e-4
WEIGHT_DECAY = 0.1
MIN_LR_RATIO = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("This training script requires a CUDA GPU.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(1337)
np.random.seed(1337)
random.seed(1337)
torch.cuda.manual_seed_all(1337)

print("Device:", DEVICE)
print("Checkpoint target:", CKPT_PATH)


# ============================================================
# DATASET & PINNED MEMORY BATCH LOADER
# ============================================================

if not os.path.exists(LOCAL_BIN_PATH):
    if not os.path.exists(BIN_PATH):
        raise FileNotFoundError(
            f"Token file not found:\n{BIN_PATH}\n"
            "Check BIN_PATH and verify file location."
        )
    print("Copying token file to local instance storage...")
    shutil.copy2(BIN_PATH, LOCAL_BIN_PATH)

data = np.memmap(
    LOCAL_BIN_PATH,
    dtype=np.uint16,
    mode="r",
)

if len(data) <= BLOCK_SIZE:
    raise ValueError(
        f"Dataset contains {len(data)} tokens, but BLOCK_SIZE is {BLOCK_SIZE}."
    )

print(f"Loaded token dataset: {len(data):,} tokens")


def get_batch(batch_size=BATCH_SIZE, block_size=BLOCK_SIZE):
    ix = torch.randint(
        low=0,
        high=len(data) - block_size,
        size=(batch_size,),
    ).tolist()

    # pin_memory enables non-blocking asynchronous DMA transfer to GPU
    x = torch.stack([
        torch.from_numpy(data[i : i + block_size].astype(np.int64))
        for i in ix
    ]).pin_memory()

    y = torch.stack([
        torch.from_numpy(data[i + 1 : i + 1 + block_size].astype(np.int64))
        for i in ix
    ]).pin_memory()

    return x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)


# ============================================================
# LEARNING RATE SCHEDULE
# ============================================================

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return current_step / max(1, WARMUP_STEPS)

    progress = (current_step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    progress = min(max(progress, 0.0), 1.0)
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))

    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine_decay


# ============================================================
# CHECKPOINT HELPERS
# ============================================================

def cpu_rng_state():
    return torch.get_rng_state().cpu().clone()


def cuda_rng_states():
    if torch.cuda.is_available():
        return [
            state.cpu().clone()
            for state in torch.cuda.get_rng_state_all()
        ]
    return None


def save_checkpoint(
    path,
    step,
    model,
    optimizer,
    scheduler,
    scaler,
    loss,
):
    checkpoint = {
        "format_version": 1,
        "step": int(step),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": float(loss),
        "config": {
            "vocab_size": VOCAB_SIZE,
            "n_layer": N_LAYER,
            "n_head": N_HEAD,
            "n_embd": N_EMBD,
            "block_size": BLOCK_SIZE,
            "num_experts": NUM_EXP,
            "top_k": TOP_K,
            "batch_size": BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "total_steps": TOTAL_STEPS,
            "warmup_steps": WARMUP_STEPS,
            "base_lr": BASE_LR,
            "weight_decay": WEIGHT_DECAY,
            "min_lr_ratio": MIN_LR_RATIO,
            "data_path": BIN_PATH,
        },
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": cpu_rng_state(),
        "cuda_rng_state_all": cuda_rng_states(),
    }

    tmp_path = path + ".tmp"
    try:
        torch.save(checkpoint, tmp_path)
        os.replace(tmp_path, path)
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


def restore_rng_states(checkpoint):
    if "python_rng_state" in checkpoint:
        random.setstate(checkpoint["python_rng_state"])

    if "numpy_rng_state" in checkpoint:
        np.random.set_state(checkpoint["numpy_rng_state"])

    if checkpoint.get("torch_rng_state") is not None:
        state = checkpoint["torch_rng_state"]
        state = state.detach().cpu().to(torch.uint8)
        torch.set_rng_state(state)

    saved_cuda_states = checkpoint.get("cuda_rng_state_all")
    if saved_cuda_states is not None and torch.cuda.is_available():
        current_device_count = torch.cuda.device_count()
        if len(saved_cuda_states) == current_device_count:
            states = [
                state.detach().cpu().to(torch.uint8)
                for state in saved_cuda_states
            ]
            torch.cuda.set_rng_state_all(states)


def load_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
):
    if not os.path.isfile(path):
        print("No checkpoint found. Starting from scratch.")
        return 0

    print(f"Loading checkpoint: {path}")
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler.load_state_dict(checkpoint.get("scaler_state_dict", {}))

    if "scheduler_state_dict" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    restore_rng_states(checkpoint)

    saved_step = int(checkpoint["step"])
    next_step = saved_step + 1

    print(f"Checkpoint step: {saved_step + 1}")
    print(f"Resuming at loop step: {next_step}")
    print("Restored learning rate:", f"{scheduler.get_last_lr()[0]:.6e}")

    del checkpoint
    return next_step


# ============================================================
# INITIALIZATION & COMPILATION
# ============================================================

base_model = CustomMoEGPT(
    vocab_size=VOCAB_SIZE,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    block_size=BLOCK_SIZE,
    num_experts=NUM_EXP,
    top_k=TOP_K,
).to(DEVICE)

optimizer = bnb.optim.PagedAdamW8bit(
    base_model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler("cuda", enabled=True)
scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)

# Load checkpoint onto uncompiled model first
start_step = load_checkpoint(
    CKPT_PATH,
    base_model,
    optimizer,
    scheduler,
    scaler,
)

try:
    model = torch.compile(base_model)
    print("torch.compile enabled successfully.")
except Exception as exc:
    print(f"torch.compile skipped; using eager execution: {exc}")
    model = base_model


# ============================================================
# TRAINING LOOP WITH THROUGHPUT PROFILING
# ============================================================

TOKENS_PER_STEP = BATCH_SIZE * BLOCK_SIZE * GRADIENT_ACCUMULATION_STEPS

model.train()

window_tokens = 0
window_time_sec = 0.0

print(f"Starting training loop from step {start_step}...")

for step in range(start_step, TOTAL_STEPS):
    # Synchronize before starting timing window to catch preceding operations
    torch.cuda.synchronize()
    step_start_time = time.perf_counter()

    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = 0.0
    accumulated_aux_loss = 0.0

    for micro_step in range(GRADIENT_ACCUMULATION_STEPS):
        x, y = get_batch()

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            logits, aux_loss = model(x)

            ce_loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )

            total_loss = ce_loss + (AUX_LOSS_COEF * aux_loss)
            loss = total_loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        accumulated_loss += ce_loss.detach().item() / GRADIENT_ACCUMULATION_STEPS
        accumulated_aux_loss += aux_loss.detach().item() / GRADIENT_ACCUMULATION_STEPS

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        base_model.parameters(),
        max_norm=1.0,
    )

    scale_before = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    scale_after = scaler.get_scale()

    if scale_after >= scale_before:
        scheduler.step()

    # Synchronize at step boundary to capture full GPU execution time
    torch.cuda.synchronize()
    step_elapsed_sec = time.perf_counter() - step_start_time

    window_time_sec += step_elapsed_sec
    window_tokens += TOKENS_PER_STEP

    completed_step = step + 1

    if completed_step % LOG_INTERVAL == 0:
        current_lr = scheduler.get_last_lr()[0]
        avg_step_ms = (window_time_sec / LOG_INTERVAL) * 1000.0
        tokens_per_sec = window_tokens / window_time_sec
        peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

        print(
            f"Step {completed_step:5d}/{TOTAL_STEPS} | "
            f"CE: {accumulated_loss:.4f} | "
            f"Aux: {accumulated_aux_loss:.4f} | "
            f"Speed: {avg_step_ms:6.1f} ms/step | "
            f"Throughput: {tokens_per_sec:6,.0f} tok/s | "
            f"LR: {current_lr:.2e} | "
            f"Peak VRAM: {peak_vram_gb:.2f} GB"
        )

        window_tokens = 0
        window_time_sec = 0.0

    if (
        completed_step % SAVE_INTERVAL == 0
        or completed_step == TOTAL_STEPS
    ):
        save_checkpoint(
            path=CKPT_PATH,
            step=step,
            model=base_model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            loss=accumulated_loss,
        )
        print(f"--> Checkpoint saved at step {completed_step}: {CKPT_PATH}")

print("Training finished.")

Device: cuda
Checkpoint target: /content/drive/MyDrive/sprog_checkpoints/tinystories_run_v6/latest.ckpt
Copying token file to local instance storage...
Loaded token dataset: 462,302,769 tokens
Loading checkpoint: /content/drive/MyDrive/sprog_checkpoints/tinystories_run_v6/latest.ckpt
Checkpoint step: 47000
Resuming at loop step: 47000
Restored learning rate: 6.524328e-05
torch.compile enabled successfully.
Starting training loop from step 47000...


W1113 17:47:48.641000 728 torch/_inductor/utils.py:1806] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Step 47100/50000 | CE: 1.2847 | Aux: 15.9975 | Speed: 1448.8 ms/step | Throughput: 22,618 tok/s | LR: 6.49e-05 | Peak VRAM: 7.59 GB
Step 47200/50000 | CE: 1.2914 | Aux: 16.0020 | Speed: 1266.2 ms/step | Throughput: 25,878 tok/s | LR: 6.46e-05 | Peak VRAM: 7.59 GB
Step 47300/50000 | CE: 1.2982 | Aux: 15.9995 | Speed: 1266.3 ms/step | Throughput: 25,877 tok/s | LR: 6.43e-05 | Peak VRAM: 7.59 GB
Step 47400/50000 | CE: 1.2668 | Aux: 16.0080 | Speed: 1265.6 ms/step | Throughput: 25,892 tok/s | LR: 6.40e-05 | Peak VRAM: 7.59 GB
Step 47500/50000 | CE: 1.2700 | Aux: 16.0053 | Speed: 1263.7 ms/step | Throughput: 25,930 tok/s | LR: 6.37e-05 | Peak VRAM: 7.60 GB
--> Checkpoint saved at step 47500: /content/drive/MyDrive/sprog_checkpoints/tinystories_run_v6/latest.ckpt
Step 47600/50000 | CE: 1.2496 | Aux: 15.9973 | Speed: 1308.6 ms/step | Throughput: 25,041 tok/s | LR: 6.34e-05 | Peak VRAM: 7.60 GB
Step 47700/50000 | CE: 1.3402 | Aux: 16.0026 | Speed: 1265.7 ms/step | Throughput: 25,889 tok/s | LR